# 4. Validação referencial

Confere se todo ingrediente referenciado em recipes/buildings existe de
fato na tabela de items (direto ou via correção de casing). Um órfão aqui
vira um ingrediente que "desaparece" silenciosamente no app.

Usa `scripts/crafting_graph.py` - a MESMA lógica de resolução/grafo que os
scripts de geração usam - em vez de reimplementar aqui. Assim a validação
roda exatamente sobre o que vai virar o artefato, sem risco de divergir.


In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
sys.path.insert(0, str(REPO_ROOT / "scripts"))

import crafting_graph as cg  # scripts/crafting_graph.py - the single source of truth

import json
import pandas as pd  # só para exibir os órfãos em tabela


## Órfãos em recipes


In [2]:
items = cg.load_rows(cg.ITEM_DT)
recipes = cg.load_rows(cg.RECIPE_DT)
buildings = cg.load_rows(cg.BUILDOBJECT_DT)
resolve = cg.build_resolver(items)

_, orphans_recipes = cg.build_item_graph(items, recipes, resolve)
print(f"{len(orphans_recipes)} referência(s) de recipe que não resolvem para nenhum item conhecido:")
pd.DataFrame(orphans_recipes, columns=["item_id", "ingredient_id_raw"])


0 referência(s) de recipe que não resolvem para nenhum item conhecido:


,item_id,ingredient_id_raw


## Órfãos em buildings


In [3]:
_, orphans_buildings = cg.build_building_graph(buildings, resolve)
print(f"{len(orphans_buildings)} referência(s) de building que não resolvem para nenhum item conhecido:")
pd.DataFrame(orphans_buildings, columns=["building_id", "material_id_raw"])


0 referência(s) de building que não resolvem para nenhum item conhecido:


,building_id,material_id_raw


## Regressão: referências penduradas no output já commitado

Mesma checagem, mas em cima de `src/data/items.json`/`buildings.json` já
gerados - garante que nada que chega no app tem uma referência sem destino.


In [4]:
committed_items = json.loads((REPO_ROOT / "src/data/items.json").read_text(encoding="utf-8"))
committed_buildings = json.loads((REPO_ROOT / "src/data/buildings.json").read_text(encoding="utf-8"))
known_ids = set(committed_items) | set(committed_buildings)

dangling = []
for source_name, db in [("items.json", committed_items), ("buildings.json", committed_buildings)]:
    for entry_id, entry in db.items():
        for ingredient_id in entry.get("ingredients", {}):
            if ingredient_id not in known_ids:
                dangling.append((source_name, entry_id, ingredient_id))

print(f"{len(dangling)} referência(s) pendurada(s) no output final já commitado:")
dangling


0 referência(s) pendurada(s) no output final já commitado:


[]